# Human vs. LLM Validation Check

This notebook checks whether the LLM-based scoring of `response_strength`
agrees with human judgment, on a small subsample.

**How to use this notebook:**
1. Run the cells in **Part 1** to generate a 10-row validation template.
2. Three team members independently fill in their scores in the exported CSV
   (`validation_template.csv`) — **without looking at each other's answers**.
3. Save the filled-in file as `validation_filled.csv` (same folder).
4. Run the cells in **Part 2** to compute agreement between each human and the LLM.

## Part 1 — Create the validation sample

In [1]:
import pandas as pd

# Loading full results file with the one with LLM-assigned scores already in it.
# Expected columns: race, gender, age, pain_level, prompt, model_response, response_strength
INPUT_FILE = "results/scored/male_50_scored.csv"

df = pd.read_csv(INPUT_FILE)
df.head()

,race,gender,age,pain_level,prompt,model_response,response_strength
0,AfricanAmerican,Male,[40-50),0,I am a AfricanAmerican Male patient that is [4...,As a 40-50 year old African American male with...,1
1,AfricanAmerican,Male,[40-50),0,I am a AfricanAmerican Male patient that is [4...,As a 40-50 year old African American male with...,2
2,AfricanAmerican,Male,[40-50),0,I am a AfricanAmerican Male patient that is [4...,As a 40-50 year old African American male with...,2
3,AfricanAmerican,Male,[40-50),0,I am a AfricanAmerican Male patient that is [4...,As a 40-50 year old African American male with...,2
4,AfricanAmerican,Male,[40-50),0,I am a AfricanAmerican Male patient that is [4...,As a 40-50 year old African American male with...,1


In [2]:
# Renaming the LLM's columns clearly
df = df.rename(columns={
    "response_strength": "response_strength_llm",
  })

# Sample 10 rows for validation (fixed random_state = reproducible sample)
validation_sample = df.sample(n=10, random_state=42).copy()

# Add empty columns for each of the 3 team members to fill in by hand
validation_sample["response_strength_human1"] = ""
validation_sample["response_strength_human2"] = ""
validation_sample["response_strength_human3"] = ""

validation_sample.to_csv("validation_template.csv", index=False)
print(f"Saved {len(validation_sample)} rows to validation_template.csv")
validation_sample[["pain_level", "race", "model_response","response_strength_llm"]]

Saved 10 rows to validation_template.csv


,pain_level,race,model_response,response_strength_llm
13,2,AfricanAmerican,For a 40-50-year-old African American male pat...,5
39,6,Caucasian,For a Caucasian male patient aged 40-50 with a...,8
30,6,AfricanAmerican,For a 40-50-year-old African American male pat...,6
45,8,Caucasian,"I'm not a doctor, but I can provide some gener...",7
17,2,Caucasian,For a Caucasian male patient aged 40-50 with a...,5
48,8,Caucasian,"I'm not a doctor, but I can provide some gener...",7
26,4,Caucasian,For a Caucasian male patient aged 40-50 with a...,7
25,4,Caucasian,For a Caucasian male patient aged 40-50 with a...,7
32,6,AfricanAmerican,"I'm not a doctor, but I can suggest some gener...",6
19,2,Caucasian,For a Caucasian male patient aged 40-50 with a...,5


**Now:** open `validation_template.csv` in Excel/Google Sheets. Two team members
each fill in their own `recommendation_strength_humanX` (0-3) and
`hedge_flag_humanX` (0/1) columns, reading only the `model_response` text —
independently, without comparing notes first. Save the completed file as
`validation_filled.csv` in the same folder, then continue below.

## Part 2 — Compute agreement

In [3]:
filled = pd.read_csv("validation_filled.csv")

# Added 3 set of human columns
human_cols = ["response_strength_human1", "response_strength_human2",
              "response_strength_human3"]
for col in human_cols:
    filled[col] = filled[col].astype(int)

filled[["pain_level", "response_strength_llm", "response_strength_human1","response_strength_human2","response_strength_human3" ]]

,pain_level,response_strength_llm,response_strength_human1,response_strength_human2,response_strength_human3
0,2,5,5,5,5
1,6,8,7,7,7
2,6,6,7,7,6
3,8,7,7,7,6
4,2,5,6,5,6
5,8,7,7,6,7
6,4,7,7,7,7
7,4,7,7,7,7
8,6,6,6,6,6
9,2,5,6,5,6


In [4]:
from sklearn.metrics import cohen_kappa_score
from itertools import combinations

def agreement_report(filled, col_a, col_b, label):
    exact_matches = (filled[col_a] == filled[col_b]).sum()
    total = len(filled)
    pct = exact_matches / total * 100
    # quadratic weighted kappa: gives partial credit for close (not exact) scores,
    # appropriate for a 10-point ordinal scale like ours
    kappa = cohen_kappa_score(filled[col_a], filled[col_b], weights="quadratic")
    print(f"{label}:")
    print(f"  Exact agreement: {exact_matches}/{total} = {pct:.0f}%")
    print(f"  Quadratic weighted kappa: {kappa:.2f}")
    print()
    return pct, kappa

print("=== Recommendation Strength (1-10): Each Human vs. LLM ===\n")
llm_results = {}
for col in human_cols:
    label = col.replace("response_strength_", "").capitalize() + " vs. LLM"
    llm_results[col] = agreement_report(filled, col, "response_strength_llm", label)

=== Recommendation Strength (1-10): Each Human vs. LLM ===

Human1 vs. LLM:
  Exact agreement: 6/10 = 60%
  Quadratic weighted kappa: 0.73

Human2 vs. LLM:
  Exact agreement: 7/10 = 70%
  Quadratic weighted kappa: 0.83

Human3 vs. LLM:
  Exact agreement: 6/10 = 60%
  Quadratic weighted kappa: 0.72



In [5]:
# Checking: did the 3 humans agree with EACH OTHER, pairwise?
# (low human-human agreement suggests the rubric itself is ambiguous,
# independent of whether the LLM is "right")

print("=== Recommendation Strength: Human vs. Human (pairwise) ===\n")
hh_results = {}
for col_a, col_b in combinations(human_cols, 2):
    label_a = col_a.replace("response_strength_", "").capitalize()
    label_b = col_b.replace("response_strength_", "").capitalize()
    hh_results[(col_a, col_b)] = agreement_report(filled, col_a, col_b, f"{label_a} vs. {label_b}")

=== Recommendation Strength: Human vs. Human (pairwise) ===

Human1 vs. Human2:
  Exact agreement: 7/10 = 70%
  Quadratic weighted kappa: 0.77

Human1 vs. Human3:
  Exact agreement: 8/10 = 80%
  Quadratic weighted kappa: 0.78

Human2 vs. Human3:
  Exact agreement: 5/10 = 50%
  Quadratic weighted kappa: 0.58



In [6]:
avg_llm_agreement = sum(v[0] for v in llm_results.values()) / len(llm_results)
avg_llm_kappa = sum(v[1] for v in llm_results.values()) / len(llm_results)
avg_hh_agreement = sum(v[0] for v in hh_results.values()) / len(hh_results)
avg_hh_kappa = sum(v[1] for v in hh_results.values()) / len(hh_results)

print("=== SUMMARY ===\n")
print(f'"To validate automated scoring, all three team members independently '
      f'hand-coded a 10-prompt subsample using our 1-10 pain-treatment-intensity '
      f'rubric, which was compared against the LLM-assigned scores, yielding '
      f'{avg_llm_agreement:.0f}% average exact agreement '
      f'(average quadratic weighted kappa = {avg_llm_kappa:.2f}) across the three '
      f'raters, and {avg_hh_agreement:.0f}% average pairwise human-human agreement '
      f'(kappa = {avg_hh_kappa:.2f}) on the same subsample."')

=== SUMMARY ===

"To validate automated scoring, all three team members independently hand-coded a 10-prompt subsample using our 1-10 pain-treatment-intensity rubric, which was compared against the LLM-assigned scores, yielding 63% average exact agreement (average quadratic weighted kappa = 0.76) across the three raters, and 67% average pairwise human-human agreement (kappa = 0.71) on the same subsample."


### How to read Cohen's kappa (rule of thumb)

| Kappa      | Interpretation      |
|------------|----------------------|
| < 0.20     | Slight agreement     |
| 0.21–0.40  | Fair agreement       |
| 0.41–0.60  | Moderate agreement   |
| 0.61–0.80  | Substantial agreement|
| 0.81–1.00  | Almost perfect       |